# PharmGraph — Pass 2: Heterogeneous GNN (GraphSAGE)

This notebook implements the heterogeneous GNN described in the ML plan (Step 5).
It uses **the identical data split and negative samples produced by Preprocessing_part2**
so that baseline (Pass 1) and GNN metrics are directly comparable.

| Component | Choice |
|-----------|--------|
| Framework | PyTorch Geometric (`torch_geometric`) |
| Graph type | `HeteroData` (typed nodes + typed edges) |
| Convolution | `HeteroConv` wrapping `SAGEConv` per relation |
| Prediction head | MLP dot-product decoder on `(gene, drug)` pairs |
| Loss | Binary cross-entropy |
| Metrics | ROC-AUC, Average Precision, Precision@k, Recall@k |

**Inputs** (from `pharmgraph_pass1_artifacts/`):
- `train_graph_edges.csv`
- `gene_drug_train_pairs_labeled.csv`
- `gene_drug_val_pairs_labeled.csv`
- `gene_drug_test_pairs_labeled.csv`
- `unified_graph_nodes.csv`

**Outputs** (written to `pass2_gnn/`):
- `gnn_metrics.csv` — same schema as `baseline_metrics.csv` for direct comparison
- `gnn_val_scored_predictions.csv`
- `gnn_test_scored_predictions.csv`
- `gnn_top_predicted_novel_gene_drug_links.csv`
- `comparison_table.csv` — baseline vs GNN side-by-side

## 0. Install dependencies

In [1]:
# Run once in Colab — kernel restart not required after this cell
import sys

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install torch_scatter torch_sparse torch_cluster torch_spline_conv \
    -f https://data.pyg.org/whl/torch-2.1.0+cu118.html
!pip install -q torch_geometric
# Got rid of pyg_lib because it wouldn't install

print("Dependencies installed.")

Looking in links: https://data.pyg.org/whl/torch-2.1.0+cu118.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 12.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 25.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for torch_scatter: filename=torch_scatter-2.1.2-cp312-cp312-linux_x86_64.whl size=3835209 sha256=0822741820860cde30a311215940b6e0f422c48932086bcd38449dc7d048b6f4
  Stored in directory: /root/.cache/pip/wheels/84/20/50/44800723f57cd798630e77b3ec83bc80bd26a1e3dc3a672ef5
  Created wheel for torch_sparse: filename=torch_sparse-0.6.18-cp312-cp312-linux_x86_64.whl size=3117973 sha256=3dd4236c8c40e56d40741c785e8fc7e46729c920c6acd201db6b9b7bad350f81
  Stored in directory: /root/.cache/pip/wheels/71/fa/21/bd1d78ce16

## 1. Imports & configuration

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# ── Paths ─────────────────────────────────────────────────────────────────────
# Point ARTIFACT_DIR at the folder produced by Preprocessing_part2

# ── Hyperparameters ───────────────────────────────────────────────────────────
RANDOM_SEED  = 42
EMBED_DIM    = 64    # initial node feature dimension (learned embedding table)
HIDDEN_DIM   = 128   # GNN hidden dimension
OUT_DIM      = 64    # final node representation dimension
NUM_LAYERS   = 2     # number of GNN message-passing layers
DROPOUT      = 0.3
LR           = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS       = 100
PATIENCE     = 10    # early stopping patience (epochs without val improvement)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device : cuda
PyTorch: 2.10.0+cu128


## 2. Load data

In [5]:
train_graph_edges = pd.read_csv("train_graph_edges.csv")
nodes_df          = pd.read_csv("unified_graph_nodes.csv")

train_pairs = pd.read_csv("gene_drug_train_pairs_labeled.csv")
val_pairs   = pd.read_csv("gene_drug_val_pairs_labeled.csv")
test_pairs  = pd.read_csv("gene_drug_test_pairs_labeled.csv")

train_pos = pd.read_csv("gene_drug_train_pos.csv")
val_pos   = pd.read_csv("gene_drug_val_pos.csv")
test_pos  = pd.read_csv("gene_drug_test_pos.csv")

# Can also load baseline metrics for comparison at the end

print("train_graph_edges:", train_graph_edges.shape)
print("nodes_df          :", nodes_df.shape)
print("train_pairs       :", train_pairs.shape)
print("val_pairs         :", val_pairs.shape)
print("test_pairs        :", test_pairs.shape)

train_graph_edges: (68122, 6)
nodes_df          : (27355, 2)
train_pairs       : (103070, 6)
val_pairs         : (17258, 6)
test_pairs        : (17684, 6)


## 3. Build integer node indices

PyTorch Geometric requires each node type to have its own 0-based integer index.
We build a mapping `node_id (string) → local integer index` **per node type**.

In [6]:
# Collect all node IDs that appear in the training graph
# (nodes not in the training graph cannot receive embeddings)
graph_node_ids = set(train_graph_edges["source_id"]) | set(train_graph_edges["target_id"])

# Filter nodes_df to only those present in the training graph
nodes_in_graph = nodes_df[nodes_df["node_id"].isin(graph_node_ids)].copy()

# Build per-type integer index maps
node_type_to_index = {}   # { node_type: { node_id: local_int_idx } }
node_type_counts   = {}   # { node_type: total_count }

for ntype, group in nodes_in_graph.groupby("node_type"):
    ids = group["node_id"].tolist()
    node_type_to_index[ntype] = {nid: i for i, nid in enumerate(ids)}
    node_type_counts[ntype]   = len(ids)

print("Node counts per type (in training graph):")
for ntype, cnt in node_type_counts.items():
    print(f"  {ntype:<12}: {cnt:,}")

Node counts per type (in training graph):
  disease     : 756
  drug        : 18,481
  gene        : 5,292
  variant     : 2,826


## 4. Build `HeteroData` graph object

`HeteroData` is PyG's container for heterogeneous graphs.
It stores node features and edge indices keyed by type.

Since we have no pre-computed node features, each node type gets a
**learned embedding table** (an `nn.Embedding`) that will be optimised
alongside the GNN weights. The embedding table is initialised here as a
placeholder integer index tensor; the actual parameter matrix lives in the model.

In [7]:
# ── Helper: convert a pair of node_id strings to a [2, E] edge index tensor ──
def make_edge_index(src_ids, tgt_ids, src_type, tgt_type):
    src_map = node_type_to_index[src_type]
    tgt_map = node_type_to_index[tgt_type]

    src_idx = [src_map[s] for s in src_ids if s in src_map and tgt_ids[list(src_ids).index(s)] in tgt_map]
    tgt_idx = [tgt_map[t] for t in tgt_ids if src_ids[list(tgt_ids).index(t)] in src_map and t in tgt_map]

    # Robust version: iterate together
    pairs = [(src_map[s], tgt_map[t])
             for s, t in zip(src_ids, tgt_ids)
             if s in src_map and t in tgt_map]

    if not pairs:
        return torch.zeros((2, 0), dtype=torch.long)

    src_t, tgt_t = zip(*pairs)
    return torch.tensor([list(src_t), list(tgt_t)], dtype=torch.long)


# ── Build HeteroData ─────────────────────────────────────────────────────────
data = HeteroData()

# Node features: just integer indices (embedding table in model will handle them)
for ntype, cnt in node_type_counts.items():
    data[ntype].x = torch.arange(cnt, dtype=torch.long)
    data[ntype].num_nodes = cnt

# Edge indices — one per relation type
RELATION_SPECS = [
    # (relation_type_str,  src_node_type, tgt_node_type)
    ("gene_drug",    "gene",    "drug"),
    ("gene_disease", "gene",    "disease"),
    ("variant_drug", "variant", "drug"),
    ("variant_gene", "variant", "gene"),
    # Reverse edges — important for message passing back to gene and drug nodes
    ("drug_gene",    "drug",    "gene"),
    ("disease_gene", "disease", "gene"),
    ("drug_variant", "drug",    "variant"),
    ("gene_variant", "gene",    "variant"),
]

FORWARD_RELATIONS = {
    "gene_drug":    ("gene",    "drug"),
    "gene_disease": ("gene",    "disease"),
    "variant_drug": ("variant", "drug"),
    "variant_gene": ("variant", "gene"),
}
REVERSE_MAP = {
    "gene_drug":    "drug_gene",
    "gene_disease": "disease_gene",
    "variant_drug": "drug_variant",
    "variant_gene": "gene_variant",
}

for rel, (src_type, tgt_type) in FORWARD_RELATIONS.items():
    subset = train_graph_edges[train_graph_edges["relation_type"] == rel]
    ei = make_edge_index(
        subset["source_id"].tolist(),
        subset["target_id"].tolist(),
        src_type, tgt_type
    )
    data[src_type, rel, tgt_type].edge_index = ei

    # Add reverse
    rev_rel = REVERSE_MAP[rel]
    data[tgt_type, rev_rel, src_type].edge_index = ei.flip(0)

    print(f"  {src_type} --[{rel}]--> {tgt_type}: {ei.shape[1]:,} edges")

print("\nHeteroData built successfully.")
print(data)

  gene --[gene_drug]--> drug: 51,535 edges
  gene --[gene_disease]--> disease: 9,425 edges
  variant --[variant_drug]--> drug: 4,332 edges
  variant --[variant_gene]--> gene: 2,830 edges

HeteroData built successfully.
HeteroData(
  disease={
    x=[756],
    num_nodes=756,
  },
  drug={
    x=[18481],
    num_nodes=18481,
  },
  gene={
    x=[5292],
    num_nodes=5292,
  },
  variant={
    x=[2826],
    num_nodes=2826,
  },
  (gene, gene_drug, drug)={ edge_index=[2, 51535] },
  (drug, drug_gene, gene)={ edge_index=[2, 51535] },
  (gene, gene_disease, disease)={ edge_index=[2, 9425] },
  (disease, disease_gene, gene)={ edge_index=[2, 9425] },
  (variant, variant_drug, drug)={ edge_index=[2, 4332] },
  (drug, drug_variant, variant)={ edge_index=[2, 4332] },
  (variant, variant_gene, gene)={ edge_index=[2, 2830] },
  (gene, gene_variant, variant)={ edge_index=[2, 2830] }
)


## 5. Prepare training pairs as tensors

The GNN produces node embeddings. The link prediction head takes a
`(gene_embedding, drug_embedding)` pair and outputs a score.
Here we convert the labeled pair DataFrames into index tensors and label tensors.

In [8]:
gene_map = node_type_to_index["gene"]
drug_map = node_type_to_index["drug"]


def pairs_to_tensors(pairs_df):
    """
    Convert a labeled pairs DataFrame to:
      gene_idx : LongTensor [N]
      drug_idx : LongTensor [N]
      labels   : FloatTensor [N]
    Rows where either endpoint is not in the training graph are dropped.
    Returns tensors and a mask of valid rows.
    """
    valid_mask = (
        pairs_df["source_id"].isin(gene_map) &
        pairs_df["target_id"].isin(drug_map)
    )
    valid = pairs_df[valid_mask].copy()
    dropped = (~valid_mask).sum()
    if dropped > 0:
        print(f"  Dropped {dropped} pairs (node not in training graph)")

    gene_idx = torch.tensor([gene_map[g] for g in valid["source_id"]], dtype=torch.long)
    drug_idx = torch.tensor([drug_map[d] for d in valid["target_id"]], dtype=torch.long)
    labels   = torch.tensor(valid["label"].values, dtype=torch.float)

    return gene_idx, drug_idx, labels, valid


print("Train pairs:")
train_gene_idx, train_drug_idx, train_labels, train_valid = pairs_to_tensors(train_pairs)
print(f"  {len(train_gene_idx):,} pairs")

print("Val pairs:")
val_gene_idx, val_drug_idx, val_labels, val_valid = pairs_to_tensors(val_pairs)
print(f"  {len(val_gene_idx):,} pairs")

print("Test pairs:")
test_gene_idx, test_drug_idx, test_labels, test_valid = pairs_to_tensors(test_pairs)
print(f"  {len(test_gene_idx):,} pairs")

Train pairs:
  103,070 pairs
Val pairs:
  17,258 pairs
Test pairs:
  17,684 pairs


## 6. Define the heterogeneous GNN model

**Architecture overview:**

```
Input: per-type learned embedding tables
  ↓
HeteroConv (SAGEConv per relation)  ×  NUM_LAYERS
  ↓
gene_emb [N_gene, OUT_DIM]   drug_emb [N_drug, OUT_DIM]
  ↓
Link prediction head: MLP( concat(gene_emb, drug_emb) ) → scalar score
```

**Why SAGEConv?**  
GraphSAGE aggregates neighbour features with mean pooling and concatenates
with the target node's own features. This is well-suited to our graph because
genes and drugs have highly variable degree, and SAGEConv handles inductive
settings (unseen nodes at inference) more gracefully than spectral methods.

**Why reverse edges?**  
Message passing only flows in the direction of the edge index. Without reverse
edges, drug nodes cannot receive messages from the gene nodes they're connected
to, making drug embeddings uninformed by the graph context. Adding reverse
relations makes message passing bidirectional.

In [9]:
class HeteroGNN(nn.Module):
    def __init__(
        self,
        node_type_counts: dict,   # { node_type: num_nodes }
        relation_specs: list,     # [ (rel_name, src_type, tgt_type), ... ]
        embed_dim: int = 64,
        hidden_dim: int = 128,
        out_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
    ):
        super().__init__()

        # ── Per-type embedding tables (learnable node features) ──────────────
        self.embeddings = nn.ModuleDict({
            ntype: nn.Embedding(cnt, embed_dim)
            for ntype, cnt in node_type_counts.items()
        })

        # ── GNN layers ────────────────────────────────────────────────────────
        # Each layer is a HeteroConv that applies one SAGEConv per relation type
        self.convs = nn.ModuleList()
        in_dim = embed_dim

        for layer_i in range(num_layers):
            out = out_dim if layer_i == num_layers - 1 else hidden_dim
            conv_dict = {}
            for rel_name, src_type, tgt_type in relation_specs:
                conv_dict[(src_type, rel_name, tgt_type)] = SAGEConv(
                    in_channels=(in_dim, in_dim),
                    out_channels=out,
                )
            self.convs.append(HeteroConv(conv_dict, aggr="sum"))
            in_dim = out

        self.dropout = nn.Dropout(dropout)

        # ── Link prediction head ──────────────────────────────────────────────
        # Takes concatenated gene + drug embeddings and outputs a single logit
        self.link_head = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(out_dim, 1),
        )

    def encode(self, data):
        """
        Run message passing and return a dict of node embeddings per type.
        data[ntype].x holds integer indices into the embedding table.
        """
        # Initialise x_dict from embedding tables
        x_dict = {
            ntype: self.embeddings[ntype](data[ntype].x)
            for ntype in self.embeddings
        }

        # Apply GNN layers
        for conv in self.convs:
            x_dict = conv(x_dict, data.edge_index_dict)
            # ReLU + dropout after each layer
            x_dict = {k: self.dropout(F.relu(v)) for k, v in x_dict.items()}

        return x_dict

    def decode(self, x_dict, gene_idx, drug_idx):
        """
        Given encoded node embeddings, score (gene, drug) candidate pairs.
        Returns raw logits (not probabilities).
        """
        gene_emb = x_dict["gene"][gene_idx]   # [N, out_dim]
        drug_emb = x_dict["drug"][drug_idx]   # [N, out_dim]
        pair_emb = torch.cat([gene_emb, drug_emb], dim=-1)  # [N, out_dim*2]
        return self.link_head(pair_emb).squeeze(-1)          # [N]

    def forward(self, data, gene_idx, drug_idx):
        x_dict = self.encode(data)
        return self.decode(x_dict, gene_idx, drug_idx)

In [10]:
# Build relation spec list from the forward + reverse relations defined earlier
ALL_RELATION_SPECS = [
    ("gene_drug",    "gene",    "drug"),
    ("gene_disease", "gene",    "disease"),
    ("variant_drug", "variant", "drug"),
    ("variant_gene", "variant", "gene"),
    ("drug_gene",    "drug",    "gene"),
    ("disease_gene", "disease", "gene"),
    ("drug_variant", "drug",    "variant"),
    ("gene_variant", "gene",    "variant"),
]

model = HeteroGNN(
    node_type_counts=node_type_counts,
    relation_specs=ALL_RELATION_SPECS,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    out_dim=OUT_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")
print(model)

Trainable parameters: 2,022,721
HeteroGNN(
  (embeddings): ModuleDict(
    (disease): Embedding(756, 64)
    (drug): Embedding(18481, 64)
    (gene): Embedding(5292, 64)
    (variant): Embedding(2826, 64)
  )
  (convs): ModuleList(
    (0-1): 2 x HeteroConv(num_relations=8)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (link_head): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


## 7. Move data to device

In [11]:
data = data.to(DEVICE)

train_gene_idx = train_gene_idx.to(DEVICE)
train_drug_idx = train_drug_idx.to(DEVICE)
train_labels   = train_labels.to(DEVICE)

val_gene_idx = val_gene_idx.to(DEVICE)
val_drug_idx = val_drug_idx.to(DEVICE)
val_labels   = val_labels.to(DEVICE)

test_gene_idx = test_gene_idx.to(DEVICE)
test_drug_idx = test_drug_idx.to(DEVICE)
test_labels   = test_labels.to(DEVICE)

print("Data moved to", DEVICE)

Data moved to cuda


## 8. Training loop with early stopping

In [12]:
def compute_val_auc(model, data, gene_idx, drug_idx, labels):
    model.eval()
    with torch.no_grad():
        logits = model(data, gene_idx, drug_idx)
        probs  = torch.sigmoid(logits).cpu().numpy()
    return roc_auc_score(labels.cpu().numpy(), probs)


best_val_auc   = -1.0
best_state     = None
epochs_no_improve = 0
history = []

# Positive class weight for BCE: compensates for 1:1 neg ratio
# (balanced by construction so pos_weight = 1.0, but left explicit for clarity)
pos_weight = torch.tensor([1.0]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Training for up to {EPOCHS} epochs (patience={PATIENCE})...\n")

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()

    logits = model(data, train_gene_idx, train_drug_idx)
    loss   = criterion(logits, train_labels)

    loss.backward()
    optimizer.step()

    # ── Validate ─────────────────────────────────────────────────────────────
    val_auc = compute_val_auc(model, data, val_gene_idx, val_drug_idx, val_labels)

    history.append({"epoch": epoch, "train_loss": loss.item(), "val_auc": val_auc})

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>4}  loss={loss.item():.4f}  val_auc={val_auc:.4f}")

    # ── Early stopping ────────────────────────────────────────────────────────
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (best val AUC={best_val_auc:.4f})")
            break

# Restore best weights
model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
print(f"\nTraining complete. Best val ROC-AUC: {best_val_auc:.4f}")

Training for up to 100 epochs (patience=10)...

Epoch    1  loss=0.7270  val_auc=0.5209
Epoch   10  loss=0.6717  val_auc=0.6538
Epoch   20  loss=0.6251  val_auc=0.6659
Epoch   30  loss=0.5673  val_auc=0.7334
Epoch   40  loss=0.5019  val_auc=0.7712
Epoch   50  loss=0.4460  val_auc=0.7945
Epoch   60  loss=0.4115  val_auc=0.8083
Epoch   70  loss=0.3826  val_auc=0.8191
Epoch   80  loss=0.3628  val_auc=0.8276
Epoch   90  loss=0.3438  val_auc=0.8334
Epoch  100  loss=0.3282  val_auc=0.8381

Training complete. Best val ROC-AUC: 0.8381


## 9. Evaluation — same metrics as Pass 1 baseline

In [13]:
def precision_recall_at_k(y_true, y_score, k):
    """Identical implementation to Pass 1 for fair comparison."""
    order = np.argsort(-y_score)
    y_true_sorted = np.array(y_true)[order]
    topk = y_true_sorted[:k]
    precision_k = topk.mean() if len(topk) > 0 else 0.0
    total_positives = y_true_sorted.sum()
    recall_k = topk.sum() / total_positives if total_positives > 0 else 0.0
    return float(precision_k), float(recall_k)


def evaluate(model, data, gene_idx, drug_idx, labels):
    model.eval()
    with torch.no_grad():
        logits = model(data, gene_idx, drug_idx)
        probs  = torch.sigmoid(logits).cpu().numpy()
    y = labels.cpu().numpy()
    return probs, y


val_probs,  val_y  = evaluate(model, data, val_gene_idx,  val_drug_idx,  val_labels)
test_probs, test_y = evaluate(model, data, test_gene_idx, test_drug_idx, test_labels)

val_preds  = (val_probs  >= 0.5).astype(int)
test_preds = (test_probs >= 0.5).astype(int)

val_roc_auc  = roc_auc_score(val_y,  val_probs)
val_ap       = average_precision_score(val_y,  val_probs)
test_roc_auc = roc_auc_score(test_y, test_probs)
test_ap      = average_precision_score(test_y, test_probs)

print("Validation metrics")
print(f"  ROC-AUC          : {val_roc_auc:.4f}")
print(f"  Average Precision: {val_ap:.4f}")
print()
print("Test metrics")
print(f"  ROC-AUC          : {test_roc_auc:.4f}")
print(f"  Average Precision: {test_ap:.4f}")

Validation metrics
  ROC-AUC          : 0.8381
  Average Precision: 0.8265

Test metrics
  ROC-AUC          : 0.8374
  Average Precision: 0.8284


In [14]:
k_values = [10, 25, 50, 100, 250, 500]

metric_rows = []
for split_name, y_true, y_score, roc_auc, ap in [
    ("val",  val_y,  val_probs,  val_roc_auc,  val_ap),
    ("test", test_y, test_probs, test_roc_auc, test_ap),
]:
    row = {"split": split_name, "roc_auc": roc_auc, "average_precision": ap}
    for k in k_values:
        p_k, r_k = precision_recall_at_k(y_true, y_score, k)
        row[f"precision@{k}"] = p_k
        row[f"recall@{k}"]    = r_k
    metric_rows.append(row)

gnn_metrics_df = pd.DataFrame(metric_rows)
display(gnn_metrics_df)

print("\nValidation classification report")
print(classification_report(val_y,  val_preds,  digits=4))
print("Test classification report")
print(classification_report(test_y, test_preds, digits=4))

,split,roc_auc,average_precision,precision@10,recall@10,precision@25,recall@25,precision@50,recall@50,precision@100,recall@100,precision@250,recall@250,precision@500,recall@500
0,val,0.838106,0.826462,0.9,0.001043,0.96,0.002781,0.98,0.005679,0.99,0.011473,0.960,0.027813,0.954,0.055279
1,test,0.837392,0.828391,0.9,0.001018,0.96,0.002714,0.96,0.005429,0.94,0.010631,0.972,0.027482,0.974,0.055078



Validation classification report
              precision    recall  f1-score   support

         0.0     0.8510    0.5763    0.6872      8629
         1.0     0.6797    0.8991    0.7741      8629

    accuracy                         0.7377     17258
   macro avg     0.7653    0.7377    0.7307     17258
weighted avg     0.7653    0.7377    0.7307     17258

Test classification report
              precision    recall  f1-score   support

         0.0     0.8454    0.5713    0.6818      8842
         1.0     0.6762    0.8955    0.7706      8842

    accuracy                         0.7334     17684
   macro avg     0.7608    0.7334    0.7262     17684
weighted avg     0.7608    0.7334    0.7262     17684



## 10. Baseline vs GNN comparison table

In [ ]:
# def tag_model(df, model_name):
#     df = df.copy()
#     df.insert(0, "model", model_name)
#     return df

# comparison_df = pd.concat([
#     tag_model(baseline_metrics, "Node2Vec + LR"),
#     tag_model(gnn_metrics_df,   "HeteroGNN"),
# ], ignore_index=True)

# # Show test split only for brevity
# print("=== Test split comparison ===")
# display(comparison_df[comparison_df["split"] == "test"].reset_index(drop=True))

# print("\n=== Val split comparison ===")
# display(comparison_df[comparison_df["split"] == "val"].reset_index(drop=True))

## 11. Score novel gene–drug candidates

In [15]:
all_genes_in_graph = sorted([n for n in gene_map])
all_drugs_in_graph = sorted([n for n in drug_map])

known_positive_pairs = set(
    map(tuple,
        pd.concat([train_pos, val_pos, test_pos], ignore_index=True)[
            ["source_id", "target_id"]
        ].to_records(index=False)
    )
)

print(f"Genes in graph     : {len(all_genes_in_graph):,}")
print(f"Drugs in graph     : {len(all_drugs_in_graph):,}")
print(f"Known positive pairs: {len(known_positive_pairs):,}")

Genes in graph     : 5,292
Drugs in graph     : 18,481
Known positive pairs: 69,006


In [16]:
# Score in batches to stay within GPU memory
BATCH_SIZE = 100_000
TOP_K_EXPORT = 500

model.eval()

# Pre-compute all node embeddings once (efficient — no need to re-run GNN per batch)
with torch.no_grad():
    x_dict = model.encode(data)

gene_emb_all = x_dict["gene"]  # [N_gene, OUT_DIM]
drug_emb_all = x_dict["drug"]  # [N_drug, OUT_DIM]

scored_parts = []
candidate_buffer = []

n_scored = 0
for g_str in all_genes_in_graph:
    for d_str in all_drugs_in_graph:
        if (g_str, d_str) in known_positive_pairs:
            continue
        candidate_buffer.append((gene_map[g_str], drug_map[d_str], g_str, d_str))

        if len(candidate_buffer) >= BATCH_SIZE:
            g_idx_b = torch.tensor([x[0] for x in candidate_buffer], dtype=torch.long, device=DEVICE)
            d_idx_b = torch.tensor([x[1] for x in candidate_buffer], dtype=torch.long, device=DEVICE)
            with torch.no_grad():
                logits_b = model.decode(x_dict, g_idx_b, d_idx_b)
                probs_b  = torch.sigmoid(logits_b).cpu().numpy()
            for (_, _, g_s, d_s), prob in zip(candidate_buffer, probs_b):
                scored_parts.append({"source_id": g_s, "target_id": d_s, "pred_prob": float(prob)})
            n_scored += len(candidate_buffer)
            candidate_buffer = []
            print(f"  Scored {n_scored:,} pairs...", end="\r")

# Flush remaining
if candidate_buffer:
    g_idx_b = torch.tensor([x[0] for x in candidate_buffer], dtype=torch.long, device=DEVICE)
    d_idx_b = torch.tensor([x[1] for x in candidate_buffer], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        logits_b = model.decode(x_dict, g_idx_b, d_idx_b)
        probs_b  = torch.sigmoid(logits_b).cpu().numpy()
    for (_, _, g_s, d_s), prob in zip(candidate_buffer, probs_b):
        scored_parts.append({"source_id": g_s, "target_id": d_s, "pred_prob": float(prob)})
    n_scored += len(candidate_buffer)

candidate_scored_df = (
    pd.DataFrame(scored_parts)
    .sort_values("pred_prob", ascending=False)
    .reset_index(drop=True)
)

print(f"\nTotal candidates scored: {len(candidate_scored_df):,}")
display(candidate_scored_df.head(20))


Total candidates scored: 97,732,446


,source_id,target_id,pred_prob
0,gene:cyp2d6,drug:ataluren,0.998815
1,gene:nfe2l2,drug:ataluren,0.998780
2,gene:ar,drug:curcumin,0.998779
3,gene:cyp3a4,drug:ataluren,0.998705
4,gene:ehmt2,drug:ataluren,0.998691
5,gene:gabrb3,drug:ataluren,0.998668
6,gene:cyp1a2,drug:ataluren,0.998590
7,gene:a2m,drug:ataluren,0.998515
8,gene:hif1a,drug:ataluren,0.998502
9,gene:nras,drug:ataluren,0.998293


## 12. Save all outputs

In [19]:
# Scored predictions
val_scored = val_valid.copy()
val_scored["pred_prob"]      = val_probs
val_scored["pred_label_0.5"] = val_preds
val_scored.to_csv("gnn_val_scored_predictions.csv", index=False)

test_scored = test_valid.copy()
test_scored["pred_prob"]      = test_probs
test_scored["pred_label_0.5"] = test_preds
test_scored.to_csv("gnn_test_scored_predictions.csv", index=False)

# GNN metrics (same schema as baseline_metrics.csv)
gnn_metrics_df.to_csv("gnn_metrics.csv", index=False)

# Comparison table
# comparison_df.to_csv("comparison_table.csv", index=False)

# Top novel predictions
candidate_scored_df.head(TOP_K_EXPORT).to_csv("gnn_top_predicted_novel_gene_drug_links.csv", index=False)

# Training history
pd.DataFrame(history).to_csv("gnn_training_history.csv", index=False)


## 13. Final summary

In [20]:
print("=" * 60)
print("PharmGraph Pass 2 — Heterogeneous GNN Summary")
print("=" * 60)
print(f"\nModel         : HeteroGNN ({NUM_LAYERS}-layer HeteroConv / SAGEConv)")
print(f"Parameters    : {total_params:,}")
print(f"Best val AUC  : {best_val_auc:.4f}")
print()
print("Test metrics:")
print(f"  ROC-AUC          : {test_roc_auc:.4f}")
print(f"  Average Precision: {test_ap:.4f}")
for k in k_values:
    test_row = gnn_metrics_df[gnn_metrics_df["split"] == "test"].iloc[0]
    print(f"  Precision@{k:<4}  : {test_row[f'precision@{k}']:.4f}    Recall@{k}: {test_row[f'recall@{k}']:.4f}")
print()
print("Baseline comparison (test split):")
# display(comparison_df[comparison_df["split"] == "test"][
    # ["model", "roc_auc", "average_precision", "precision@50", "recall@50"]
# ].reset_index(drop=True))

PharmGraph Pass 2 — Heterogeneous GNN Summary

Model         : HeteroGNN (2-layer HeteroConv / SAGEConv)
Parameters    : 2,022,721
Best val AUC  : 0.8381

Test metrics:
  ROC-AUC          : 0.8374
  Average Precision: 0.8284
  Precision@10    : 0.9000    Recall@10: 0.0010
  Precision@25    : 0.9600    Recall@25: 0.0027
  Precision@50    : 0.9600    Recall@50: 0.0054
  Precision@100   : 0.9400    Recall@100: 0.0106
  Precision@250   : 0.9720    Recall@250: 0.0275
  Precision@500   : 0.9740    Recall@500: 0.0551

Baseline comparison (test split):
